In [1]:
import random
import pandas as pd
import numpy as np
import copy
from collections import Counter, defaultdict

In [ ]:
POPULASI = 1000
VIOLATION_COST = 100
ITERATION = 1000
MUTATION_PROB = 0.7
TOURNAMENT_SIZE = 10

In [3]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

# DICT

In [4]:
# =========================================================
# Hari
hariId = {
"Senin": 1,
"Selasa": 2,
"Rabu": 3,
"Kamis": 4,
"Jumat": 5,
}
# hariId

# ========================================================
# mencari slot tiap per hari
# key = hariId, value = jumlah slot integer
# contoh output: {1: 8, 2: 8, 3: 8, 4: 7, 5: 5}

slotPerHari = {} # dictionary kosongan
for _, row in slot_df.iterrows():
    hari = row["hari"] # ambil kolom hari saja

    if hari not in slotPerHari:
        slotPerHari[hari] = 1
    else:
        slotPerHari[hari] += 1

slotPerHari = {hariId[k]: v for k,v in slotPerHari.items()}
# print(slotPerHari)


# ========================================================
# guru dan nama
# key = guru_id, value = nama guru
guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)
# guruPengajar

# ========================================================
# mapping nama kelas dan tingkatan
# ada 27 kelas, contoh output: {1: [{'tingkatan': 7, 'nama_kelas': '7A'}], 2: [{'tingkatan': 7, 'nama_kelas': '7B'}]}
kelasDanTingkatan = defaultdict(list)
for _, row in kelas_df.iterrows():
    kelasDanTingkatan[row['kelas_id']].append({
        'tingkatan': row['tingkatan'],
        'nama_kelas': row['nama_kelas']
    })
kelasDanTingkatan = dict(kelasDanTingkatan)
# print(kelasDanTingkatan)

# ========================================================
# mapping nama mapel dan id
# key = mapel_id, value = nama mapel
namaMapelDanId = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)
# namaMapelDanId

# =========================================================
# mencari jam per minggu tiap mape
# key = mapel_id, value = jam per minggu integerl
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)
# print(jamPerMingguMapel)

# =========================================================
# mapel id dan hari MGMP
# contoh output: {1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 3, 7: 1, 8: 1, 9: 3, 10: 4, 11: 5, 12: 3, 13: 2}
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)
mgmpMapel = {mapel_id: hariId[hari] for mapel_id, hari in mgmpMapel.items()}
# print(mgmpMapel)
# =========================================================
# batas siang dan batas MGMP
# key = hariId value = slot ke berapa dalam hari tersebut
batasSiang = {1: 5, 2: 5, 3: 4, 4: 5, 5: 4}
batasMGMP = {1: 2, 2: 2, 3: 2, 4: 2, 5: 1}
# =========================================================
# durasi guru mengajar
# key = guru value = list of dict {mapel_id, tingkatan, durasi}
durasiGuruMengajar = defaultdict(list)

for _, row in relasi_guru_mapel_df.iterrows():
    durasiGuruMengajar[row["guru_id"]].append({
        "mapel_id": row["mapel_id"],
        "tingkatan": row["tingkatan"],
        "durasi": row["durasi"]
    })
durasiGuruMengajar = dict(durasiGuruMengajar)
# durasiGuruMengajar


# =========================================================
# Wali kelas guru
# key = guru_id, value = kelas_id
waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)
# waliKelas

# =========================================================
# indexKelas
kelasIndex = {}
slotRange = 0
for i in range(1, 28):
    end = slotRange + 36
    kelasIndex[i] = (slotRange, end)
    slotRange = end

kelasIndex

{1: (0, 36),
 2: (36, 72),
 3: (72, 108),
 4: (108, 144),
 5: (144, 180),
 6: (180, 216),
 7: (216, 252),
 8: (252, 288),
 9: (288, 324),
 10: (324, 360),
 11: (360, 396),
 12: (396, 432),
 13: (432, 468),
 14: (468, 504),
 15: (504, 540),
 16: (540, 576),
 17: (576, 612),
 18: (612, 648),
 19: (648, 684),
 20: (684, 720),
 21: (720, 756),
 22: (756, 792),
 23: (792, 828),
 24: (828, 864),
 25: (864, 900),
 26: (900, 936),
 27: (936, 972)}

# INDIVIDU

In [5]:
# memecah jam_per_minggu menjadi blok yang bisa didistribusikan
def blokDistribusi(jam):
    if jam == 2:
        return [2]
    if jam == 3:
        return [3]
    if jam == 4:
        return [2,2]
    if jam == 5:
        return [2,3]
    
    return [jam]

In [6]:
# mengambil guru berdsakan mapel dan tingaktan
def ambilGuruValid(mapel_id, tingkatan):
    listGuru = []

    for guru_id, relasiList in durasiGuruMengajar.items():
        for relasi in relasiList:
            if relasi["mapel_id"] == mapel_id and relasi["tingkatan"] == tingkatan:
                listGuru.append(guru_id)
    return listGuru

In [7]:
# # membuat jadwal kosongan dulu
# def jadwalKosongan():
#     jadwal = {}
#     for hari in slotPerHari.keys():
#         jadwal[hari] = []

#     return jadwal

In [8]:
# def slotTersedia(jadwal_kelas, hari, durasi):
#     slotTerpakai = 0

#     for event in jadwal_kelas[hari]:
#         slotTerpakai += event["durasi"]
    
#     if slotTerpakai + durasi <= slotPerHari[hari]:
#         return True
    
#     return False

In [9]:
# def putEvent(jadwal_kelas, event):

#     listHari = list(slotPerHari.keys())

#     random.shuffle(listHari)

#     for hari in listHari:

#         if slotTersedia(jadwal_kelas, hari, event["durasi"]):
#             jadwal_kelas[hari].append(event)
            
#             return True
        
#     return False

In [10]:
def perluasBlok(mapel_id, guru_id, durasi):

    slot = []

    for _ in range(durasi):

        slot.append((mapel_id, guru_id))

    return slot

In [11]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in jamPerMingguMapel.items():

        blok = blokDistribusi(jam)

        guruValid = ambilGuruValid(mapel_id, tingkatan)

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blok:

            slot = perluasBlok(mapel_id, guru, durasi)

            pilihan.extend(slot)

    random.shuffle(pilihan)

    return pilihan

In [12]:
def individuConstruct(kelas_id):

    tingkatan = kelasDanTingkatan[kelas_id][0]["tingkatan"]

    slots = generatePerKelas(tingkatan)

    return slots

In [13]:
def individuTrigger():

    individu = []

    for kelas_id in sorted(kelasDanTingkatan.keys()):

        slots = individuConstruct(kelas_id)

        individu.extend(slots)

    return individu

In [14]:
def populasiConstruct(POPULASI):

    populasi = []

    for _ in range(POPULASI):

        individu = individuTrigger()

        populasi.append(individu)

    return populasi

In [15]:
populasiOptimasi = populasiConstruct(POPULASI)

In [16]:
populasiOptimasi[0]

[(12, 55),
 (3, 29),
 (12, 55),
 (1, 39),
 (12, 55),
 (13, 53),
 (1, 39),
 (3, 29),
 (10, 31),
 (6, 2),
 (7, 37),
 (2, 45),
 (6, 2),
 (11, 43),
 (3, 29),
 (5, 18),
 (9, 38),
 (4, 47),
 (8, 17),
 (3, 29),
 (6, 2),
 (3, 29),
 (9, 38),
 (5, 18),
 (5, 18),
 (2, 45),
 (7, 37),
 (4, 47),
 (13, 53),
 (4, 47),
 (10, 31),
 (7, 37),
 (11, 43),
 (8, 17),
 (4, 47),
 (5, 18),
 (7, 37),
 (12, 51),
 (13, 53),
 (3, 36),
 (11, 43),
 (10, 46),
 (10, 46),
 (8, 17),
 (4, 47),
 (6, 2),
 (12, 51),
 (4, 47),
 (5, 48),
 (4, 47),
 (3, 36),
 (12, 51),
 (5, 48),
 (2, 45),
 (3, 36),
 (2, 45),
 (3, 36),
 (5, 48),
 (5, 48),
 (1, 39),
 (11, 43),
 (6, 2),
 (13, 53),
 (1, 39),
 (7, 37),
 (6, 2),
 (7, 37),
 (9, 38),
 (4, 47),
 (9, 38),
 (3, 36),
 (8, 17),
 (5, 27),
 (4, 19),
 (3, 22),
 (3, 22),
 (6, 44),
 (2, 33),
 (12, 55),
 (9, 42),
 (12, 55),
 (13, 53),
 (7, 37),
 (11, 37),
 (5, 27),
 (8, 23),
 (1, 31),
 (2, 33),
 (7, 37),
 (5, 27),
 (5, 27),
 (3, 22),
 (12, 55),
 (4, 19),
 (4, 19),
 (3, 22),
 (13, 53),
 (1, 31),
 (

# EVAL

In [17]:
### Pre Eval
slotAwalHari = {}

index = 0

for hari, jumlah in slotPerHari.items():
    slotAwalHari[hari] = index
    index += jumlah


slotKeHari = {}

index = 0

for hari, jumlah in slotPerHari.items():
    for _ in range(jumlah):
        slotKeHari[index] = hari
        index += 1

In [18]:
def guruBentrok(individu):

    pelanggaran = 0

    slotPerKelas = 36
    jumlahKelas = len(kelasIndex)

    for slot in range(slotPerKelas):

        guruMengajar = []

        for kelas in range(jumlahKelas):

            index = kelas * slotPerKelas + slot

            guru = individu[index][1]

            guruMengajar.append(guru)

        if len(guruMengajar) != len(set(guruMengajar)):
            pelanggaran += 1

    return pelanggaran
# def guruBentrok(individu):
#     pelanggaran = 0

#     for hari in slotPerHari:
#         totalSlot = slotPerHari[hari]

#         for slot in range(totalSlot):
#             guruMengajar = []

#             for kelas in individu:
#                 if slot < len(individu[kelas][hari]):
#                     guru = individu[kelas][hari][slot]["guru"]
#                     guruMengajar.append(guru)

#                 if len(guruMengajar) != len(set(guruMengajar)):
#                     pelanggaran += 1
#     return pelanggaran



In [19]:
def distribusiMapel(individu):

    pelanggaran = 0

    for kelas_id,(start,end) in kelasIndex.items():

        distribusi = {}

        kelasSlots = individu[start:end]

        mapelSekarang = kelasSlots[0][0]
        count = 1

        blok = []

        for i in range(1,len(kelasSlots)):

            mapel = kelasSlots[i][0]

            if mapel == mapelSekarang:
                count += 1
            else:
                blok.append((mapelSekarang,count))
                mapelSekarang = mapel
                count = 1

        blok.append((mapelSekarang,count))

        for mapel,durasi in blok:

            if mapel not in distribusi:
                distribusi[mapel] = []

            distribusi[mapel].append(durasi)

        for mapel in distribusi:

            jam = jamPerMingguMapel[mapel]

            if jam == 2:
                if distribusi[mapel] != [2]:
                    pelanggaran += 1

            elif jam == 3:
                if distribusi[mapel] != [3]:
                    pelanggaran += 1

            elif jam == 4:
                if sorted(distribusi[mapel]) != [2,2]:
                    pelanggaran += 1

            elif jam == 5:
                if sorted(distribusi[mapel]) != [2,3]:
                    pelanggaran += 1

    return pelanggaran

# def putBlokMapel(slotHari):
#     blok = []

#     mapelSekarang = slotHari[0]["mapel"]

#     count = 1

#     for i in range(1, len(slotHari)):
#         if slotHari[i]["mapel"] == mapelSekarang:
#             count += 1
#         else:
#             blok.append((mapelSekarang, count))

#             mapelSekarang = slotHari[i]["mapel"]
#             count = 1
    
#     blok.append((mapelSekarang, count))

#     return blok

# def distribusiMapel(individu):
#     pelanggaran = 0

#     for kelas in individu:
#         distribusi = {}

#         for hari in individu[kelas]:
#             blok = putBlokMapel(individu[kelas][hari])

#             for mapel, durasi in blok:

#                 if mapel not in distribusi:
#                     distribusi[mapel] = []

#                     distribusi[mapel].append(durasi)
#         for mapel in distribusi:
#             jam = jamPerMingguMapel[mapel]

#             if jam == 2:
#                 if distribusi[mapel] != [2]:
#                     pelanggaran += 1
#             elif jam == 3:
#                 if distribusi[mapel] != [3]:
#                     pelanggaran += 1
#             elif jam == 4:
#                 if sorted(distribusi[mapel]) != [2,2]:
#                     pelanggaran += 1
#             elif jam == 5:
#                 if sorted(distribusi[mapel]) != [2,3]:
#                     pelanggaran += 1
#     return pelanggaran

In [20]:
def mapelSiang(individu):

    pelanggaran = 0

    for index,(mapel,guru) in enumerate(individu):

        slotDalamKelas = index % 36

        hari = slotKeHari[slotDalamKelas]

        slotHari = slotDalamKelas - slotAwalHari[hari]

        if mapel == 8 and slotHari > batasSiang[hari]:
            pelanggaran += 1

    return pelanggaran
# def mapelSiang(individu):
#     pelanggaran = 0

#     for kelas in individu:

#         for hari in individu[kelas]:
#             batas = batasSiang[hari]

#             for slot in range(len(individu[kelas][hari])):
#                 mapel = individu[kelas][hari][slot]["mapel"]

#                 # if mapel == 8 and slot >= batas:
#                 if mapel == 8 and slot > batas:

#                     pelanggaran += 1
#     return pelanggaran

In [21]:
def durasiGuru(individu):

    pelanggaran = 0

    loadGuru = {}

    for mapel,guru in individu:

        if guru not in loadGuru:
            loadGuru[guru] = 0

        loadGuru[guru] += 1

    for guru in loadGuru:

        if loadGuru[guru] > 40:
            pelanggaran += loadGuru[guru] - 40

    return pelanggaran
# def durasiGuru(individu):
#     pelanggaran = 0

#     loadGuru = {}

#     for kelas in individu:
#         for hari in individu[kelas]:

#             for slot in individu[kelas][hari]:

#                 guru = slot["guru"]

#                 if guru not in loadGuru:
#                     loadGuru[guru] = 0

#                 loadGuru[guru] += 1
#     for guru in loadGuru:
#         if loadGuru[guru] > 40:
#             pelanggaran += loadGuru[guru] - 40
#     return pelanggaran

In [22]:
def waktuMGMP(individu):

    pelanggaran = 0

    for index,(mapel,guru) in enumerate(individu):

        if mapel in mgmpMapel:

            slotDalamKelas = index % 36

            hari = slotKeHari[slotDalamKelas]

            slotHari = slotDalamKelas - slotAwalHari[hari]

            if hari == mgmpMapel[mapel]:

                if slotHari > batasMGMP[hari]:

                    pelanggaran += 1

    return pelanggaran
# def waktuMGMP(individu):
#     pelanggaran = 0

#     for kelas in individu:

#         for hari in individu[kelas]:
#             for slot in range(len(individu[kelas][hari])):
#                 mapel = individu[kelas][hari][slot]["mapel"]

#                 if mapel in mgmpMapel:
#                     hariMGMP = mgmpMapel[mapel]

#                     if hari == hariMGMP:
#                         if slot > batasMGMP[hari]:
#                             pelanggaran += 1

#     return pelanggaran

In [23]:
def cekWaliKelas(individu):

    pelanggaran = 0

    for kelas_id,(start,end) in kelasIndex.items():

        for i in range(start,end):

            guru = individu[i][1]

            if guru in waliKelas:

                if kelas_id != waliKelas[guru]:

                    pelanggaran += 1

    return pelanggaran
# def cekWaliKelas(individu):
#     pelanggaran = 0

#     for kelas_id, jadwalKelas in individu.items():

#         for hari in jadwalKelas:

#             for slot in jadwalKelas[hari]:

#                 guru = slot['guru']

#                 if guru in waliKelas:

#                     kelasWali = waliKelas[guru]

#                     if kelas_id != kelasWali:
#                         pelanggaran += 1
#     return pelanggaran

In [24]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)

    pelanggaran += distribusiMapel(individu)

    pelanggaran += mapelSiang(individu)

    pelanggaran += durasiGuru(individu)

    pelanggaran += waktuMGMP(individu)

    pelanggaran += cekWaliKelas(individu)

    return pelanggaran * VIOLATION_COST

# CACHE

In [25]:
fitnessCache = {}

def hashIndividu(individu):
    return tuple(individu)

In [26]:
def evaluasiCache(individu):

    key = hashIndividu(individu)

    if key in fitnessCache:
        return fitnessCache[key]
    
    fitness = evaluasiIndividu(individu)

    fitnessCache[key] = fitness

    return fitness

# PROBLEM SLOT

In [27]:
def slotBermasalah(individu):

    masalah = []

    slotPerkelas = 36
    jumlahKelas = len(kelasIndex)

    for slot in range(slotPerkelas):

        guruMengajar = {}

        for kelas in range(jumlahKelas):

            index = kelas * slotPerkelas + slot

            mapel, guru = individu[index]

            if guru in guruMengajar:

                masalah.append(index)
            else:
                guruMengajar[guru] = index
    
    return masalah

# GA

In [28]:
def kelasBermasalah(individu):

    konflik = {}

    for kelas,(start,end) in kelasIndex.items():

        konflik[kelas] = 0

        for i in range(start,end):

            if i in slotBermasalah(individu):

                konflik[kelas] += 1

    return max(konflik,key=konflik.get)

def crossover(parent1, parent2):

    child = parent1.copy()

    kelas = random.choice(list(kelasIndex.keys()))

    start, end = kelasIndex[kelas]

    child[start:end] = parent2[start:end]

    return child

def crossoverTargeted(parent1,parent2):

    child = parent1.copy()

    kelas = kelasBermasalah(parent1)

    start,end = kelasIndex[kelas]

    child[start:end] = parent2[start:end]

    return child

# def crossover(parent1, parent2):
#     child1 = copy.deepcopy(parent1)
#     child2 = copy.deepcopy(parent2)

#     hari = list(parent1.keys())

#     # pilih hari yang akan ditukar
#     jumlah = random.randint(1, len(hari)//2)

#     hariTerpilih = random.sample(hari, jumlah)

#     for h in hariTerpilih:
#         child1[h], child2[h] = parent2[h], parent1[h]
        
#     return child1, child2

In [29]:
def clusterMapel(slot):
    kelompok = defaultdict(list)

    for mapel,guru in slot:
        kelompok[mapel].append((mapel,guru))

    hasil = []

    for mapel in kelompok:
        hasil.extend(kelompok[mapel])

    return hasil

def mutasi(individu, MUTATION_PROB):

    child = individu.copy()

    if random.random() > MUTATION_PROB:
        return child

    kelas = random.choice(list(kelasIndex.keys()))

    start, end = kelasIndex[kelas]

    slotKelas = child[start:end]

    slotKelas = clusterMapel(slotKelas)

    child[start:end] = slotKelas

    return child

def mutasiTargeted(individu):

    child = individu.copy()

    masalah = slotBermasalah(child)

    if not masalah:
        return child

    i = random.choice(masalah)

    j = random.randint(0,len(child)-1)

    child[i],child[j] = child[j],child[i]

    return child

# def mutasi(individu, MUTATION_PROB):

#     individuBaru = copy.deepcopy(individu)

#     if random.random() > MUTATION_PROB:
#         return individuBaru

#     hari = random.choice(list(individuBaru.keys()))

#     slot = individuBaru[hari]

#     # jika slot adalah dict
#     if isinstance(slot, dict):
#         keys = list(slot.keys())

#         if len(keys) < 2:
#             return individuBaru

#         i, j = random.sample(keys, 2)

#         slot[i], slot[j] = slot[j], slot[i]

#     # jika slot adalah list
#     else:
#         if len(slot) < 2:
#             return individuBaru

#         i, j = random.sample(range(len(slot)), 2)

#         slot[i], slot[j] = slot[j], slot[i]

#     return individuBaru

In [30]:
def turnamen(populasi, fitnessPop, TOURNAMENT_SIZE):

    kandidat =random.sample(range(len(populasi)), TOURNAMENT_SIZE)

    terbaik = kandidat[0]

    for i in kandidat:

        if fitnessPop[i] < fitnessPop[terbaik]:
            terbaik = i
    
    return populasi[terbaik]
# def turnamen(populasi, TOURNAMENT_SIZE):
#     kandidat = random.sample(populasi, TOURNAMENT_SIZE)

#     terbaik = min(kandidat, key=lambda x: evaluasiIndividu(x))

#     return terbaik

# MAIN

In [31]:
def geneticAlgorithm(populasiAwal, GENERASI, POPULASI, TOURNAMENT_SIZE, MUTATION_PROB):

    populasi = populasiAwal

    bestIndividu = None
    bestFitness = float("inf")

    for gen in range(GENERASI):

        # =============================
        # Evaluasi fitness populasi
        # =============================
        fitnessPop = []

        for individu in populasi:

            fitness = evaluasiCache(individu)

            fitnessPop.append(fitness)

            if fitness < bestFitness:
                bestFitness = fitness
                bestIndividu = individu

        print("Generasi:", gen, "Best Fitness:", bestFitness)

        # =============================
        # Elitism (menyimpan individu terbaik)
        # =============================
        elitIndex = sorted(range(len(fitnessPop)), key=lambda i: fitnessPop[i])

        elit = [populasi[i] for i in elitIndex[:2]]

        # =============================
        # Membuat populasi baru
        # =============================
        populasiBaru = elit.copy()

        while len(populasiBaru) < POPULASI:

            # -------------------------
            # Selection
            # -------------------------
            parent1 = turnamen(populasi, fitnessPop, TOURNAMENT_SIZE)
            parent2 = turnamen(populasi, fitnessPop, TOURNAMENT_SIZE)

            # -------------------------
            # Crossover
            # -------------------------
            if random.random() < 0.5:
                child = crossover(parent1, parent2)
            else:
                child = crossoverTargeted(parent1, parent2)

            # -------------------------
            # Mutation
            # -------------------------
            if random.random() < MUTATION_PROB:

                if random.random() < 0.5:
                    child = mutasi(child, MUTATION_PROB)
                else:
                    child = mutasiTargeted(child)

            populasiBaru.append(child)

        populasi = populasiBaru

    return bestIndividu, bestFitness

In [32]:
hasil = geneticAlgorithm(populasiOptimasi, ITERATION, POPULASI, TOURNAMENT_SIZE, MUTATION_PROB)

Generasi: 0 Best Fitness: 90700
Generasi: 1 Best Fitness: 89700
Generasi: 2 Best Fitness: 87500
Generasi: 3 Best Fitness: 86500
Generasi: 4 Best Fitness: 85000
Generasi: 5 Best Fitness: 82500
Generasi: 6 Best Fitness: 82000
Generasi: 7 Best Fitness: 80100
Generasi: 8 Best Fitness: 78800
Generasi: 9 Best Fitness: 77200
Generasi: 10 Best Fitness: 75500
Generasi: 11 Best Fitness: 73900
Generasi: 12 Best Fitness: 71900
Generasi: 13 Best Fitness: 70700
Generasi: 14 Best Fitness: 69400
Generasi: 15 Best Fitness: 67700
Generasi: 16 Best Fitness: 66000
Generasi: 17 Best Fitness: 64500
Generasi: 18 Best Fitness: 63700
Generasi: 19 Best Fitness: 62300
Generasi: 20 Best Fitness: 61100
Generasi: 21 Best Fitness: 60100
Generasi: 22 Best Fitness: 58900
Generasi: 23 Best Fitness: 58800
Generasi: 24 Best Fitness: 57600
Generasi: 25 Best Fitness: 56800
Generasi: 26 Best Fitness: 56800
Generasi: 27 Best Fitness: 56100
Generasi: 28 Best Fitness: 55600
Generasi: 29 Best Fitness: 54800
Generasi: 30 Best Fi